In [1]:
# -*- coding: utf-8 -*-
# ener_all.csv を最終的に使う列順に整形し、ener_all_reformatted.csv を出力します。
# "noise_rate" と "noise_rate_contribution" は
# それぞれ operation_fidelity, operation_fidelity_contribution として扱います。

import pandas as pd
from pathlib import Path

# ---- 設定 ----
# 入力/出力の場所を現在のディレクトリ構成から自動解決
CANDIDATE_ROOTS = [
    Path('.'),
    Path('..') / 'datas',
    Path('..').parent / 'pb-vqe2' / 'data' / 'figdata',
]
# 入力
_src = None
for r in CANDIDATE_ROOTS:
    p = r / 'ener_all.csv'
    if p.exists():
        _src = p
        break
SRC = _src if _src is not None else Path('ener_all.csv')
# 出力は datas/ 配下にまとめる
OUT_DIR = Path('..') / 'datas'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT = OUT_DIR / 'ener_all_reformatted.csv'

def read_csv_smart(path: Path) -> pd.DataFrame:
    """文字コードを良きに計らって読むヘルパ"""
    last_err = None
    for enc in ("utf-8", "utf-8-sig", "cp932", "shift_jis"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last_err = e
    raise last_err

# ---- 読み込み ----
src_df = read_csv_smart(SRC)

# 目標の列順（不要な列は出力しない）
target_cols = [
    "segment_id",
    "coherence_time",
    "entanglement_fidelity",
    "operation_fidelity",                 # noise_rate を置換して扱う
    "target_metric",
    "coherence_time_contribution",
    "entanglement_fidelity_contribution",
    "operation_fidelity_contribution",    # noise_rate_contribution を置換して扱う
    "use_log_scale",
]

# 出力データフレームを構築
out_df = pd.DataFrame(index=src_df.index)
for col in target_cols:
    if col == "operation_fidelity":
        # 元の operation_fidelity を優先。無ければ noise_rate を使う
        if "operation_fidelity" in src_df.columns:
            out_df[col] = src_df["operation_fidelity"]
        elif "noise_rate" in src_df.columns:
            out_df[col] = src_df["noise_rate"]
        else:
            out_df[col] = pd.NA
    elif col == "operation_fidelity_contribution":
        # 元の operation_fidelity_contribution を優先。無ければ noise_rate_contribution を使う
        if "operation_fidelity_contribution" in src_df.columns:
            out_df[col] = src_df["operation_fidelity_contribution"]
        elif "noise_rate_contribution" in src_df.columns:
            out_df[col] = src_df["noise_rate_contribution"]
        else:
            out_df[col] = pd.NA
    else:
        out_df[col] = src_df[col] if col in src_df.columns else pd.NA

# 保存（BOM付きUTF-8: Excel でも文字化けしにくい）
out_df.to_csv(OUT, index=False, encoding="utf-8-sig")

print(f"Done: wrote {OUT}")
print("Columns:", list(out_df.columns))


Done: wrote ../datas/ener_all_reformatted.csv
Columns: ['segment_id', 'coherence_time', 'entanglement_fidelity', 'operation_fidelity', 'target_metric', 'coherence_time_contribution', 'entanglement_fidelity_contribution', 'operation_fidelity_contribution', 'use_log_scale']


In [2]:
# time 側のCSVを同様に整形（各行ごとに gate_speed_factor と gate_time_ns の「中身がある方」を採用）
from pathlib import Path
import pandas as pd

# ---- 設定 ----
# 入力/出力を自動解決
CANDIDATE_ROOTS = [
    Path('.'),
    Path('..') / 'datas',
    Path('..').parent / 'pb-vqe2' / 'data' / 'figdata',
]
_src_t = None
for r in CANDIDATE_ROOTS:
    p = r / 'time_future.csv'
    if p.exists():
        _src_t = p
        break
SRC_T = _src_t if _src_t is not None else Path('time_future.csv')
OUT_DIR = Path('..') / 'datas'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_T = OUT_DIR / 'time_future_reformatted.csv'

# ---- 読み込み ----
src_t_df = read_csv_smart(SRC_T)

def _series(name: str):
    return src_t_df[name] if name in src_t_df.columns else pd.Series([pd.NA] * len(src_t_df), index=src_t_df.index)

# 列名をデータの「中身がある方」に寄せる（全行での非欠損数で判定、同数なら gate_time_ns 系を優先）
_gs = _series("gate_speed_factor")
_gt = _series("gate_time_ns")
_gs_cnt = int(_gs.notna().sum())
_gt_cnt = int(_gt.notna().sum())
var_col_name = "gate_time_ns" if _gt_cnt >= _gs_cnt else "gate_speed_factor"

_gsc = _series("gate_speed_factor_contribution")
_gtc = _series("gate_time_ns_contribution")
_gsc_cnt = int(_gsc.notna().sum())
_gtc_cnt = int(_gtc.notna().sum())
contrib_col_name = (
    "gate_time_ns_contribution" if _gtc_cnt >= _gsc_cnt else "gate_speed_factor_contribution"
)

# 目標の列順（timing_mode は出力しない）
target_cols_t = [
    "segment_id",
    "distance",
    "entanglement_speed_factor",
    var_col_name,
    "target_metric",
    "distance_contribution",
    "entanglement_speed_factor_contribution",
    contrib_col_name,
    "use_log_scale",
]

out_t_df = pd.DataFrame(index=src_t_df.index)
for col in target_cols_t:
    if col == var_col_name:
        # 列名に合わせて優先度を切り替えつつ、行ごとに非欠損を採用
        if var_col_name == "gate_speed_factor":
            out_t_df[col] = _series("gate_speed_factor").combine_first(_series("gate_time_ns"))
        else:
            out_t_df[col] = _series("gate_time_ns").combine_first(_series("gate_speed_factor"))
    elif col == contrib_col_name:
        if contrib_col_name == "gate_speed_factor_contribution":
            out_t_df[col] = _series("gate_speed_factor_contribution").combine_first(_series("gate_time_ns_contribution"))
        else:
            out_t_df[col] = _series("gate_time_ns_contribution").combine_first(_series("gate_speed_factor_contribution"))
    else:
        out_t_df[col] = src_t_df[col] if col in src_t_df.columns else pd.NA

# 保存（BOM付きUTF-8）
OUT_T.parent.mkdir(parents=True, exist_ok=True)
out_t_df.to_csv(OUT_T, index=False, encoding="utf-8-sig")

print(f"Done: wrote {OUT_T}")
print("Columns:", list(out_t_df.columns))


FileNotFoundError: [Errno 2] No such file or directory: 'time_future.csv'

In [ ]:
# time_all 側のCSVが来たときに整形するセル（存在しない場合はスキップしてメッセージ表示）
from pathlib import Path
import pandas as pd

# 入力/出力の自動解決（time_all3.csv が無ければ time_all.csv を探す）
CANDIDATE_ROOTS = [
    Path('.'),
    Path('..') / 'datas',
    Path('..').parent / 'pb-vqe2' / 'data' / 'figdata',
]
_src_t = None
for fname in ('time_all3.csv','time_all.csv'):
    for r in CANDIDATE_ROOTS:
        p = r / fname
        if p.exists():
            _src_t = p
            break
    if _src_t is not None:
        break
SRC_T = _src_t if _src_t is not None else Path('time_all3.csv')
OUT_DIR = Path('..') / 'datas'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_T = OUT_DIR / 'time_all_reformatted.csv'

if not SRC_T.exists():
    print(f"Skip: {SRC_T} not found.")
else:
    src_t_df = read_csv_smart(SRC_T)

    def _series(name: str):
        return src_t_df[name] if name in src_t_df.columns else pd.Series([pd.NA] * len(src_t_df), index=src_t_df.index)

    # 列名をデータの「中身がある方」に寄せる（全行での非欠損数で判定、同数なら gate_time_ns 系を優先）
    _gs = _series("gate_speed_factor")
    _gt = _series("gate_time_ns")
    _gs_cnt = int(_gs.notna().sum())
    _gt_cnt = int(_gt.notna().sum())
    var_col_name = "gate_time_ns" if _gt_cnt >= _gs_cnt else "gate_speed_factor"

    _gsc = _series("gate_speed_factor_contribution")
    _gtc = _series("gate_time_ns_contribution")
    _gsc_cnt = int(_gsc.notna().sum())
    _gtc_cnt = int(_gtc.notna().sum())
    contrib_col_name = (
        "gate_time_ns_contribution" if _gtc_cnt >= _gsc_cnt else "gate_speed_factor_contribution"
    )

    # 目標の列順（timing_mode は出力しない）
    target_cols_t = [
        "segment_id",
        "distance",
        "entanglement_speed_factor",
        var_col_name,
        "target_metric",
        "distance_contribution",
        "entanglement_speed_factor_contribution",
        contrib_col_name,
        "use_log_scale",
    ]

    out_t_df = pd.DataFrame(index=src_t_df.index)
    for col in target_cols_t:
        if col == var_col_name:
            # 列名に合わせて優先度を切り替えつつ、行ごとに非欠損を採用
            if var_col_name == "gate_speed_factor":
                out_t_df[col] = _series("gate_speed_factor").combine_first(_series("gate_time_ns"))
            else:
                out_t_df[col] = _series("gate_time_ns").combine_first(_series("gate_speed_factor"))
        elif col == contrib_col_name:
            if contrib_col_name == "gate_speed_factor_contribution":
                out_t_df[col] = _series("gate_speed_factor_contribution").combine_first(_series("gate_time_ns_contribution"))
            else:
                out_t_df[col] = _series("gate_time_ns_contribution").combine_first(_series("gate_speed_factor_contribution"))
        else:
            out_t_df[col] = src_t_df[col] if col in src_t_df.columns else pd.NA

    OUT_T.parent.mkdir(parents=True, exist_ok=True)
    out_t_df.to_csv(OUT_T, index=False, encoding="utf-8-sig")

    print(f"Done: wrote {OUT_T}")
    print("Columns:", list(out_t_df.columns))


# datas/ CSV quick viewer
datas 直下の CSV をプレビュー/簡易可視化します。

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt

DATAS = Path('..') / 'datas'

def list_csvs(limit: int = 50):
    files = [p for p in DATAS.glob('*.csv')]
    print(f'Found {len(files)} CSV(s) under {DATAS}')
    for p in files[:limit]:
        print('-', p.name)
    return files

try:
    read_csv_smart
except NameError:
    def read_csv_smart(path: Path) -> pd.DataFrame:
        last_err = None
        for enc in ('utf-8', 'utf-8-sig', 'cp932', 'shift_jis'):
            try:
                return pd.read_csv(path, encoding=enc)
            except Exception as e:
                last_err = e
        raise last_err

def _lim_with_pad(arr: np.ndarray, pad_ratio: float = 0.05):
    a = np.asarray(arr, dtype=float)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return (0.0, 1.0)
    lo, hi = float(np.min(a)), float(np.max(a))
    if hi <= lo:
        return (lo * 0.9 if lo != 0 else -1.0, hi * 1.1 if hi != 0 else 1.0)
    pad = (hi - lo) * pad_ratio
    return (lo - pad, hi + pad)

def visualize_3d_basic(df: pd.DataFrame, param_names, color_col: str, *, use_log_scale=True, invert_x_axis=False, invert_y_axis=False, title=None):
    xcol, ycol, zcol = param_names
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')
    def _to_log_safe(s: pd.Series):
        a = s.astype(float).values
        minpos = np.min(a[a > 0]) * 0.1 if np.any(a > 0) else 1e-12
        a = np.where(a > 0, a, minpos)
        return np.log10(a)
    if use_log_scale:
        xs = _to_log_safe(df[xcol])
        ys = _to_log_safe(df[ycol])
        zs = _to_log_safe(df[zcol])
        ax.set_xlabel(f'log10({xcol})')
        ax.set_ylabel(f'log10({ycol})')
        ax.set_zlabel(f'log10({zcol})')
    else:
        xs = df[xcol].astype(float).values
        ys = df[ycol].astype(float).values
        zs = df[zcol].astype(float).values
        ax.set_xlabel(xcol)
        ax.set_ylabel(ycol)
        ax.set_zlabel(zcol)
    color = df[color_col].astype(float).values if color_col in df.columns else np.zeros(len(df))
    sc = ax.scatter(xs, ys, zs, c=color, cmap='viridis', s=40, alpha=0.95)
    if color_col in df.columns:
        cb = fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.1)
        cb.set_label(color_col)
    ax.set_xlim(*_lim_with_pad(xs)); ax.set_ylim(*_lim_with_pad(ys)); ax.set_zlim(*_lim_with_pad(zs))
    if invert_x_axis: ax.invert_xaxis()
    if invert_y_axis: ax.invert_yaxis()
    ax.grid(True); ax.view_init(elev=20, azim=55)
    if title: plt.title(title)
    plt.tight_layout(); plt.show()

def _detect_3d_params(df: pd.DataFrame):
    cols = set(df.columns)
    # Time-like
    if {'distance','entanglement_speed_factor'}.issubset(cols) and ('gate_time_ns' in cols or 'gate_speed_factor' in cols):
        third = 'gate_time_ns' if 'gate_time_ns' in cols else 'gate_speed_factor'
        return ['distance','entanglement_speed_factor', third], 'target_metric', True
    # Energy-like
    if 'coherence_time' in cols and ('entanglement_fidelity' in cols or 'entanglement_error' in cols):
        ef = 'entanglement_fidelity' if 'entanglement_fidelity' in cols else 'entanglement_error'
        for t in ('operation_fidelity','operation_error','noise_rate'):
            if t in cols:
                return ['coherence_time', ef, t], 'target_metric', True
    # Fallback: first numeric columns
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if len(num_cols) >= 3:
        color = num_cols[3] if len(num_cols) > 3 else num_cols[0]
        return num_cols[:3], color, False
    return None, None, False

def visualize_csv_generic(csv_path: Path):
    df = read_csv_smart(csv_path)
    print(f'Read: {csv_path} (rows={len(df)}, cols={len(df.columns)})')
    display(df.head(8))
    params, color_col, logscale = _detect_3d_params(df)
    if params:
        print(f'3D params detected: {params} | color={color_col} | log={logscale}')
        visualize_3d_basic(df, params, color_col, use_log_scale=logscale, invert_x_axis=True, invert_y_axis=False)
    else:
        # Fallback: histograms of up to 6 numeric columns
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        n = min(6, len(num_cols))
        if n == 0:
            print('No numeric columns to plot.')
            return
        fig, axes = plt.subplots(1, n, figsize=(4*n, 3))
        if n == 1: axes = [axes]
        for ax, c in zip(axes, num_cols[:n]):
            ax.hist(df[c].dropna().astype(float), bins=40, color='#4C72B0', alpha=0.8)
            ax.set_title(c)
        plt.tight_layout(); plt.show()


In [ ]:
# 一覧と簡易実行例
files = list_csvs()
CSV_PATH = files[0] if files else None
if CSV_PATH is not None:
    visualize_csv_generic(CSV_PATH)
else:
    print('No CSV found under datas/. Place files and rerun.')
